In [48]:
import numpy as np
import pandas as pd
import os
import kagglehub
import warnings
warnings.filterwarnings("ignore")

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples

In [49]:
path = kagglehub.dataset_download("algozee/teenager-menthal-healy")

In [50]:
print('Arquivos do Dataset: ', os.listdir(path))

Arquivos do Dataset:  ['Teen_Mental_Health_Dataset.csv']


# Visão Geral do Dataset

Esse Dataset estuda como o uso de redes sociais afeta a saúde mental de adolescentes. Ele inclui hábitos diários como tempo gasto em redes sociais, sono, estresse, ansiedade e atividade física.

O objetivo é entender se o uso elevado de redes sociais está relacionado a problemas como estresse, ansiedade e depressão. Os dados ajudam na análise de comportamento e na construção de modelos de machine learning para prever riscos à saúde mental.

# ─────────────────────────────────────────────
# 1. CARREGAMENTO & TRANSFORMAÇÃO DE DADOS BRUTOS
# ─────────────────────────────────────────────

In [51]:
rota = os.path.join(path, 'Teen_Mental_Health_Dataset.csv')
df = pd.read_csv(rota)
df.shape

(1200, 13)

In [52]:
df.head()

,age,gender,daily_social_media_hours,platform_usage,sleep_hours,screen_time_before_sleep,academic_performance,physical_activity,social_interaction_level,stress_level,anxiety_level,addiction_level,depression_label
0,14,male,7.9,Instagram,7.4,2.9,3.01,1.5,low,2,2,1,0
1,19,female,1.9,TikTok,8.0,2.9,3.22,0.8,high,8,1,10,0
2,17,female,1.3,Instagram,7.6,0.5,3.92,0.0,high,2,4,2,0
3,15,male,7.4,TikTok,6.9,1.6,3.48,0.8,medium,1,7,9,0
4,15,female,4.7,Both,4.9,3.0,2.37,1.4,medium,3,5,2,0


In [53]:
#Converter as strings em números
interaction_map = {'low': 0, 'medium': 1, 'high': 2}
df['social_interaction_level'] = (
    df['social_interaction_level'].str.strip().str.lower().map(interaction_map)
)

le = LabelEncoder()
df['gender'] = le.fit_transform(df['gender'].str.strip())
df['platform_usage'] = le.fit_transform(df['platform_usage'].str.strip())

In [54]:
df.head()

,age,gender,daily_social_media_hours,platform_usage,sleep_hours,screen_time_before_sleep,academic_performance,physical_activity,social_interaction_level,stress_level,anxiety_level,addiction_level,depression_label
0,14,1,7.9,1,7.4,2.9,3.01,1.5,0,2,2,1,0
1,19,0,1.9,2,8.0,2.9,3.22,0.8,2,8,1,10,0
2,17,0,1.3,1,7.6,0.5,3.92,0.0,2,2,4,2,0
3,15,1,7.4,2,6.9,1.6,3.48,0.8,1,1,7,9,0
4,15,0,4.7,0,4.9,3.0,2.37,1.4,1,3,5,2,0


In [55]:
#8 horas de sono, pois é saudável + 0 que garante que a diferença não resulte em negativo (8-10 = -2)
df['deficit_sono'] = np.maximum(0, 8 - df['sleep_hours'])
#o 1e-9 garante que caso alguém tenha colocado 0 horas de sono, não de erro, e divida por um número mínimo, como 0.0000000001
df['relacao_tela_sono'] = df['screen_time_before_sleep'] / (df['sleep_hours'] + 1e-9)
#fazer a média dos pontos negativos
df['pontuacao_carga_mental'] = (
    df['stress_level'] + df['anxiety_level'] + df['addiction_level']
) / 3

In [56]:
df.head()

,age,gender,daily_social_media_hours,platform_usage,sleep_hours,screen_time_before_sleep,academic_performance,physical_activity,social_interaction_level,stress_level,anxiety_level,addiction_level,depression_label,deficit_sono,relacao_tela_sono,pontuacao_carga_mental
0,14,1,7.9,1,7.4,2.9,3.01,1.5,0,2,2,1,0,0.6,0.391892,1.666667
1,19,0,1.9,2,8.0,2.9,3.22,0.8,2,8,1,10,0,0.0,0.362500,6.333333
2,17,0,1.3,1,7.6,0.5,3.92,0.0,2,2,4,2,0,0.4,0.065789,2.666667
3,15,1,7.4,2,6.9,1.6,3.48,0.8,1,1,7,9,0,1.1,0.231884,5.666667
4,15,0,4.7,0,4.9,3.0,2.37,1.4,1,3,5,2,0,3.1,0.612245,3.333333


In [57]:
X = df.drop('depression_label', axis=1)
y = df['depression_label']

print(f'Formato de X: {X.shape}')
print(f'Colunas: {X.columns.tolist()}\n')


Formato de X: (1200, 15)
Colunas: ['age', 'gender', 'daily_social_media_hours', 'platform_usage', 'sleep_hours', 'screen_time_before_sleep', 'academic_performance', 'physical_activity', 'social_interaction_level', 'stress_level', 'anxiety_level', 'addiction_level', 'deficit_sono', 'relacao_tela_sono', 'pontuacao_carga_mental']



# ─────────────────────────────────────────────
# 2. PADRONIZAÇÃO DOS NÚMEROS
# ─────────────────────────────────────────────


**Fórmula matemática para o `fit_transform(X)`**

$$z = \frac{x - μ}{σ}$$

Ela pega cada valor (x), subtrai a média daquela coluna (μ) e divide pelo desvio padrão (σ).

In [58]:
nivelar = StandardScaler()
X_nivelado = nivelar.fit_transform(X)

print('Média após nivelar: ', X_nivelado.mean(axis=0).round(3))
print('Desvio Padrão após nivelar: ', X_nivelado.std(axis=0).round(3))

Média após nivelar:  [ 0.  0. -0.  0. -0.  0. -0. -0.  0. -0. -0. -0.  0.  0.  0.]
Desvio Padrão após nivelar:  [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


# ─────────────────────────────────────────────
# 3. PCA — VARIÂNCIA ACUMULADA
# ─────────────────────────────────────────────

Como o dataset tem muitas colunas, essa técnica vai reduzir para `super colunas`

In [59]:
reduz_dados = PCA()
reduz_dados.fit(X_nivelado)

#explained_variance -> diz quanto cada super coluna capturou da variancia da tabela toda
#vai pegar o valor de explained... e vai somar uma por uma: 1 super coluna, depois 1 + 2, depois 1+2+3...n
variancia = np.cumsum(reduz_dados.explained_variance_ratio_)

print("\nComponentes | Variância individual | Variância acumulada")
print("  " + "-" * 55)
for i, (individual, acumulada) in enumerate(
    zip(reduz_dados.explained_variance_ratio_, variancia), 1):
        mark = " ← escolhido" if i == 2 else (" ← 80%" if abs(acumulada - 0.80) < 0.03 else "")
        print(f"PC{i:2d}        |  {individual:.4f}  ({individual*100:.1f}%)       |  {acumulada:.4f}  ({acumulada*100:.1f}%) {mark}")
        if i == 10:
            break



Componentes | Variância individual | Variância acumulada
  -------------------------------------------------------
PC 1        |  0.1654  (16.5%)       |  0.1654  (16.5%) 
PC 2        |  0.1342  (13.4%)       |  0.2996  (30.0%)  ← escolhido
PC 3        |  0.1027  (10.3%)       |  0.4023  (40.2%) 
PC 4        |  0.0726  (7.3%)       |  0.4749  (47.5%) 
PC 5        |  0.0716  (7.2%)       |  0.5464  (54.6%) 
PC 6        |  0.0696  (7.0%)       |  0.6160  (61.6%) 
PC 7        |  0.0680  (6.8%)       |  0.6840  (68.4%) 
PC 8        |  0.0677  (6.8%)       |  0.7517  (75.2%) 
PC 9        |  0.0646  (6.5%)       |  0.8162  (81.6%)  ← 80%
PC10        |  0.0627  (6.3%)       |  0.8790  (87.9%) 


vamos usar 30% devido trabalhar com 2 dimensões, ficando mais fácil de enxergar

In [60]:
reduz_dados2 = PCA(n_components=2, random_state=42)
X_pca = reduz_dados2.fit_transform(X_nivelado)
X_pca.shape

(1200, 2)

In [61]:
nomes = X.columns.tolist()
receita_pca = pd.DataFrame(
    reduz_dados2.components_.T,
    index=nomes,
    columns=['PC1', 'PC2']
)

#sinal serve para dizer o quanto da determianda coluna afeta a super coluna
#o PC1 aumenta se o deficit de sono aumenta, assim como o PC1 diminui se as horas de sono aumentam
print("Como o PCA é Calculado — PC1 (Eixo do Sono):")
top_pc1 = receita_pca["PC1"].abs().sort_values(ascending=False).head(5)
for feat, val in top_pc1.items():
    sinal = "+" if receita_pca.loc[feat, "PC1"] > 0 else "-"
    print(f"{sinal} {feat:35s} |loading| = {val:.3f}")

Como o PCA é Calculado — PC1 (Eixo do Sono):
+ deficit_sono                        |loading| = 0.537
- sleep_hours                         |loading| = 0.535
+ relacao_tela_sono                   |loading| = 0.526
+ screen_time_before_sleep            |loading| = 0.300
+ pontuacao_carga_mental              |loading| = 0.160


In [62]:
print("Como o PCA é Calculado — PC2 (Eixo da Carga Mental):")
top_pc2 = receita_pca["PC2"].abs().sort_values(ascending=False).head(5)
for feat, val in top_pc2.items():
    sinal = "+" if receita_pca.loc[feat, "PC2"] > 0 else "-"
    print(f"{sinal} {feat:35s} |loading| = {val:.3f}")

Como o PCA é Calculado — PC2 (Eixo da Carga Mental):
+ pontuacao_carga_mental              |loading| = 0.681
+ anxiety_level                       |loading| = 0.417
+ stress_level                        |loading| = 0.398
+ addiction_level                     |loading| = 0.382
- relacao_tela_sono                   |loading| = 0.143


In [63]:
print(f"PC1: {reduz_dados2.explained_variance_ratio_[0]*100:.1f}% -> Eixo do Sono")
print(f"PC2: {reduz_dados2.explained_variance_ratio_[1]*100:.1f}% -> Eixo da Carga Mental")
print(f"Total: {sum(reduz_dados2.explained_variance_ratio_)*100:.1f}%\n")

PC1: 16.5% -> Eixo do Sono
PC2: 13.4% -> Eixo da Carga Mental
Total: 30.0%



# ─────────────────────────────────────────────
# 4. ESCOLHA DO K
# ─────────────────────────────────────────────

* K -> Número de Grupos que vai ser criado
* Inércia -> Mede o quão compactos estão os Grupos
* Silhueta -> Mede o quão parecido o adolescente é com o seu próprio grupo e o quão diferente ele é dos outros grupos

Quanto mais grupos eu crio, menor fica a inércia e a silhueta tem uma nota que vai de -1 a 1, sendo mais próximos do 1 = mais perfeitos e bem separados estão os grupos

In [64]:
melhor_sil_k, melhor_sil = 2,0
inercias, silhouettes = [],[]

print(f'\n{'k':>3} | {'Inércia':>10} | {'Silhouette':>10} | Barra')
print("  " + "-" * 55)

for k in range(2,9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    grupo = km.fit_predict(X_nivelado) #quem foi pra qual grupo
    inercia = km.inertia_
    sil = silhouette_score(X_nivelado, grupo)
    inercias.append(inercia)
    silhouettes.append(sil)
    bar  = "█" * int(sil * 200) #vai deixar mais visual
    mark = " ← melhor" if sil > melhor_sil else ""
    if sil > melhor_sil:
        melhor_sil, melhor_sil_k = sil, k
    print(f'{k:>3} | {inercia:>10.1f} | {sil:10.4f} | {bar}{mark}')
print(f'\nK bom pela Silhueta: K = {melhor_sil_k}')


  k |    Inércia | Silhouette | Barra
  -------------------------------------------------------
  2 |    15842.0 |     0.1143 | ██████████████████████ ← melhor
  3 |    14740.0 |     0.0931 | ██████████████████
  4 |    13936.1 |     0.0919 | ██████████████████
  5 |    13504.0 |     0.0809 | ████████████████
  6 |    13092.4 |     0.0764 | ███████████████
  7 |    12799.1 |     0.0728 | ██████████████
  8 |    12543.8 |     0.0697 | █████████████

K bom pela Silhueta: K = 2


Para dados médicos, usaremos 4 grupos, e não 2, pois podemos ter casos: quem está ótimo, quem está levemente estressado, quem tem depressão leve e quem está em caso crítico

# ─────────────────────────────────────────────
# 5. K-MEANS
# ─────────────────────────────────────────────

In [65]:
km_final = KMeans(n_clusters=4, random_state=42, n_init=20) #n_init garante que não acerte de primeira por sorte
grupos = km_final.fit_predict(X_nivelado) #quantidade de grupos

df['cluster'] = grupos
df['PC1'] = X_pca[:, 0]
df['PC2'] = X_pca[:, 1]
df['depression_label'] = y

sil_final = silhouette_score(X_nivelado, grupos)
print(f'Pontuação do Silhueta: {sil_final:.4f}')
print(f'Inércia: {km_final.inertia_:1f}')

Pontuação do Silhueta: 0.0922
Inércia: 13934.983879


# ─────────────────────────────────────────────
# 6. NOMEAÇÃO CLÍNICA
# ─────────────────────────────────────────────

In [66]:
df.columns.tolist()

['age',
 'gender',
 'daily_social_media_hours',
 'platform_usage',
 'sleep_hours',
 'screen_time_before_sleep',
 'academic_performance',
 'physical_activity',
 'social_interaction_level',
 'stress_level',
 'anxiety_level',
 'addiction_level',
 'depression_label',
 'deficit_sono',
 'relacao_tela_sono',
 'pontuacao_carga_mental',
 'cluster',
 'PC1',
 'PC2']

In [67]:
#focado em comportamento dos adolescentes
colunas = [
    'daily_social_media_hours', 'sleep_hours', 'deficit_sono',
    'stress_level', 'anxiety_level', 'addiction_level',
    'physical_activity', 'screen_time_before_sleep', 'pontuacao_carga_mental',
    'depression_label'
]

perfil = df.groupby('cluster')[colunas].mean().round(2)
perfil['quantidade_adol'] = df['cluster'].value_counts().sort_index()
perfil['porcentagem'] = (perfil['quantidade_adol'] / len(df) * 100).round(1)

* Grupo 0: sono bom (~7.6h), stress alto (6.7), ansiedade alta (7.1) → Sobrecarga Mental
* Grupo 1: sono bom (~7.5h), stress baixo (4.0), ansiedade baixa (3.8) → Saudável
* Grupo 2: sono ruim (~5.1h), screen antes de dormir alto (2.4h) → Sono Comprometido
* Grupo 3: sono ruim (~5.2h), maior taxa de depressão (6%) → Risco

In [68]:
nomes_cluster = {
    0: 'Sobrecarga Mental',
    1: 'Saudável',
    2: 'Sono Comprometido',
    3: 'Risco'
}

print()
for grupo, linha in perfil.iterrows():
    nome = nomes_cluster[grupo]
    dep_rate = linha['depression_label'] * 100
    print(f"Grupo {grupo} — {nome}")
    print(f"{'─'*55}")
    print(f"Tamanho:              {int(linha['quantidade_adol'])} adolescentes ({linha['porcentagem']}%)")
    print(f"Horas de RS/dia:      {linha['daily_social_media_hours']:.2f}h")
    print(f"Sono:                 {linha['sleep_hours']:.2f}h  (déficit: {linha['deficit_sono']:.2f}h)")
    print(f"Estresse:               {linha['stress_level']:.2f}/10")
    print(f"Ansiedade:            {linha['anxiety_level']:.2f}/10")
    print(f"Vício (addiction):    {linha['addiction_level']:.2f}/10")
    print(f"Carga mental total:   {linha['pontuacao_carga_mental']:.2f}/10")
    print(f"Atividade física:     {linha['physical_activity']:.2f}")
    print(f"Tempo de Tela antes:  {linha['screen_time_before_sleep']:.2f}h")
    print(f"Taxa de depressão:    {dep_rate:.1f}%")
    print()


Grupo 0 — Sobrecarga Mental
───────────────────────────────────────────────────────
Tamanho:              337 adolescentes (28.1%)
Horas de RS/dia:      4.73h
Sono:                 7.60h  (déficit: 0.58h)
Estresse:               6.68/10
Ansiedade:            7.12/10
Vício (addiction):    6.88/10
Carga mental total:   6.89/10
Atividade física:     1.03
Tempo de Tela antes:  1.75h
Taxa de depressão:    1.0%

Grupo 1 — Saudável
───────────────────────────────────────────────────────
Tamanho:              303 adolescentes (25.2%)
Horas de RS/dia:      4.44h
Sono:                 7.52h  (déficit: 0.64h)
Estresse:               4.01/10
Ansiedade:            3.83/10
Vício (addiction):    3.66/10
Carga mental total:   3.83/10
Atividade física:     1.02
Tempo de Tela antes:  1.67h
Taxa de depressão:    0.0%

Grupo 2 — Sono Comprometido
───────────────────────────────────────────────────────
Tamanho:              290 adolescentes (24.2%)
Horas de RS/dia:      4.60h
Sono:                 5.13h  

# ─────────────────────────────────────────────
# 7. SILHOUETTE POR AMOSTRA
# ─────────────────────────────────────────────

In [72]:
#em vez de calcular a média, vamos calcular a nota de cada adolescente individualmente
sil_amostras = silhouette_samples(X_nivelado, grupos)
df['silhueta'] = sil_amostras

print()
sil_por_grupo = df.groupby('cluster')['silhueta'].agg(['mean', 'std', 'min', 'max'])
sil_por_grupo.index = [f'Grupo {i} - {nomes_cluster[i]}' for i in sil_por_grupo.index]
print(sil_por_grupo.round(4).to_string())


                               mean     std     min     max
Grupo 0 - Sobrecarga Mental  0.0864  0.0577 -0.0015  0.2510
Grupo 1 - Saudável           0.0983  0.0630  0.0049  0.2648
Grupo 2 - Sono Comprometido  0.1120  0.0619  0.0056  0.2601
Grupo 3 - Risco              0.0712  0.0548 -0.0097  0.2081
